# Did the conditioning adapter actually train?
Loads the trained checkpoint and runs 3 checks:
1. **Adapter moved** off its identity/no-op init.
2. **No-op invariance is now BROKEN** — conditioning changes the output (it didn't at init).
3. **PDE residual is lower** with conditioning than without (the metric the effect lives in).

A flat *noise* loss during training is expected (frozen converged base + no-op-init adapter); the real signal is here.

In [7]:
import sys, os
REPO_ROOT = "/home/rhautier/ddpm-jax"
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

import yaml, subprocess
import jax, jax.numpy as jnp
from src.models.model import DDPM, identity_combine_init
from src.train_ddpm import load_dataset
from src.utils import load_checkpoint
from src.physics_guidance import make_dx_func, make_residual_loss

# --- which run to inspect ---
CONFIG = "configs/config.yaml"          # the run you trained (Re=1000 adapter)
cfg = yaml.safe_load(open(CONFIG))
cfg["conditioning"]["train"]["enabled"] = True   # force ConditionalUnet for the diagnostic
run_name = cfg["monitoring"]["run_name"]
re = cfg["conditioning"]["train"]["re"]
ch = cfg["model"]["ch"]
n = cfg["data"]["image_size"]
print("run:", run_name, "| re:", re)

run: conditioned_frozen_base | re: 1000


In [8]:
# Model + data (for mean/std and a real sample) + cond_func
ddpm = DDPM(cfg)
model = ddpm.unet
train_ds, _, _, mean, std = load_dataset(cfg)
x0 = jnp.asarray(next(iter(train_ds)).numpy())[:1]   # one real normalized triplet (1,256,256,3)
cond_func = make_dx_func(n=n, re=re, std=float(std), mean=float(mean), lam=1.0)
print("sample:", x0.shape, "| mean=%.4f std=%.4f" % (mean, std))

Using cached flow-data/kf_2d_re1000_256_40seed.npy
Loaded flow-data/kf_2d_re1000_256_40seed.npy — shape: (40, 320, 256, 256)
Dataset split — train: 32 seqs, val: 4 seqs, test: 4 seqs
Triplets per seq: 318  |  mean=-0.0000, std=4.7988
sample: (1, 256, 256, 3) | mean=-0.0000 std=4.7988


In [9]:
# Load the latest trained checkpoint from the run's subfolder
run_dir = f"{cfg['checkpointing']['checkpoint_dir'].rstrip('/')}/{run_name}"
out = subprocess.run(["gcloud", "storage", "ls", run_dir + "/"], capture_output=True, text=True).stdout
ckpts = sorted(l.strip() for l in out.splitlines() if l.strip().endswith(".pkl"))
assert ckpts, f"no checkpoints found in {run_dir}"
print("checkpoints:", [c.split('/')[-1] for c in ckpts])
trained, _, ep = load_checkpoint(ckpts[-1])
print("loaded", ckpts[-1].split('/')[-1], "(epoch", ep, ")")

checkpoints: ['ckpt_epoch_0009.pkl', 'ckpt_epoch_0019.pkl', 'ckpt_epoch_0029.pkl']


/snap/google-cloud-cli/473/lib/third_party/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
Copying gs://ddpm-thesis-rh/checkpoints/ddpm/conditioned_frozen_base/ckpt_epoch_0029.pkl to file:///tmp/tmp4jdquuml.pkl
  
..

Average throughput: 152.4MiB/s


loaded ckpt_epoch_0029.pkl (epoch 29 )


In [10]:
# CHECK 1 — adapter moved off its init
ident = identity_combine_init(ch)(None, (1, 1, 2 * ch, ch))
d_combine = float(jnp.abs(trained["cond_combine"]["kernel"] - ident).max())
print(f"cond_combine moved from identity by: {d_combine:.3e}   (>0 expected)")
print(f"cond_in    max|w|: {float(jnp.abs(trained['cond_in']['kernel']).max()):.3e}")
print(f"cond_hidden max|w|: {float(jnp.abs(trained['cond_hidden']['kernel']).max()):.3e}")
assert d_combine > 0, "cond_combine never moved -> adapter did not train (check the optimizer freeze labels)"

cond_combine moved from identity by: 2.099e-01   (>0 expected)
cond_in    max|w|: 1.639e+00
cond_hidden max|w|: 3.838e-01


In [11]:
# CHECK 2 — no-op invariance must now BREAK (conditioning changes the output)
key = jax.random.PRNGKey(0)
t_val = 100
abar = ddpm.alpha_bar[t_val]
eps = jax.random.normal(key, x0.shape)
x_t = jnp.sqrt(abar) * x0 + jnp.sqrt(1 - abar) * eps          # noised sample at t
t_arr = jnp.array([t_val])

out_none = model.apply({"params": trained}, x_t, t_arr, train=False, condRes=None)
out_cond = model.apply({"params": trained}, x_t, t_arr, train=False, condRes=cond_func(x_t))
delta = float(jnp.abs(out_cond - out_none).max())
print(f"max |eps_cond - eps_uncond| = {delta:.3e}   (was ~0 at init; >0 means conditioning is used)")
assert delta > 1e-4, "conditioning has NO effect post-training -> condRes not reaching the model / adapter dead"

max |eps_cond - eps_uncond| = 5.748e-01   (was ~0 at init; >0 means conditioning is used)


In [12]:
# CHECK 3 (full reverse sampling) — residual of the FULLY denoised x0, conditioned vs unconditional.
# Same model + same noise both runs; cond_strength toggles the path:
#   0  -> pure conditional   (eps_cond)
#   -1 -> pure unconditional (eps_uncond; the conditional term cancels)
res_loss = make_residual_loss(n=n, re=re, std=float(std), mean=float(mean))
t_start = 300                       # reverse from a moderately-noised x0 (method regime; T=1000)
key3 = jax.random.PRNGKey(1)

cfg["conditioning"]["inference"]["cond_strength"] = 0      # conditional
x0_cond = ddpm.sample(trained, (n, n, 3), key3, float(std), float(mean), x_g=x0[0], t_start=t_start)

cfg["conditioning"]["inference"]["cond_strength"] = -1     # unconditional (base path)
x0_unc  = ddpm.sample(trained, (n, n, 3), key3, float(std), float(mean), x_g=x0[0], t_start=t_start)

r_gt   = float(res_loss(x0).mean())
r_unc  = float(res_loss(x0_unc[None]).mean())
r_cond = float(res_loss(x0_cond[None]).mean())
print(f"residual  ground-truth     : {r_gt:.4e}")
print(f"residual  unconditional x0 : {r_unc:.4e}")
print(f"residual  conditioned   x0 : {r_cond:.4e}")
print(f"--> conditioning {'LOWERS' if r_cond < r_unc else 'does NOT lower'} the residual "
      f"({100*(r_unc - r_cond)/r_unc:+.1f}% vs unconditional)")

KeyError: 'traing'